# タグを付けて訳文の文体を制御する（丁寧さの制御）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/controlled_translation.ipynb)

NLP クックブック 26-14 ／ 機械翻訳

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

同じ原文を、敬体（です・ます）と常体（だ・である）に訳し分けたい。原文の先頭に付けたタグで、訳文の文体を指定できるようにする。

In [1]:
!pip install -q datasets sentencepiece sacremoses accelerate

In [2]:
import re
from datasets import Dataset, load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

POLITE = re.compile(r"(です|ます|でした|ました|ません|でしょう|ください|ましょう)(か|よ|ね|わ)?[。！？!?]*$")
PLAIN = re.compile(r"(だ|である|だった|ない|た|る|う|い)(よ|ね|な|ぞ|わ|か)?[。！？!?]*$")

def style(ja):
    """訳文の文末から文体を判定する（規則による自動ラベル付け）"""
    return "<polite>" if POLITE.search(ja) else "<plain>" if PLAIN.search(ja) else None

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")
rows = []
for ex in ds["train"].select(range(150000)):
    en, ja = ex["translation"]["en"].strip(), ex["translation"]["ja"].strip()
    if 10 <= len(ja) <= 50 and 3 <= len(en.split()) <= 20 and style(ja):
        rows.append({"en": f"{style(ja)} {en}", "ja": ja, "style": style(ja)})
train = Dataset.from_list(rows).shuffle(seed=0)
polite = train.filter(lambda r: r["style"] == "<polite>").select(range(3000))
plain = train.filter(lambda r: r["style"] == "<plain>").select(range(3000))
from datasets import concatenate_datasets
train = concatenate_datasets([polite, plain]).shuffle(seed=0)      # 敬体と常体を同数にする
print(train[0], train[1])

{'en': '<plain> Better. But your vowels are still a bit iffy. [ normal voice ] my vowels are a little bit iffy.', 'ja': '巻き舌だからな\u3000もう終わろう', 'style': '<plain>'} {'en': '<plain> Innocence Now account was cleaned out this morning.', 'ja': 'イノセンス・ナウの口座は 今朝一掃されたわ', 'style': '<plain>'}


In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn", tgt_lang="jpn_Jpan")
model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
JA = tokenizer.convert_tokens_to_ids("jpn_Jpan")
tokenizer.add_tokens(["<polite>", "<plain>"], special_tokens=True)       # タグを新しいトークンとして追加し、埋め込みの行を増やす
model.resize_token_embeddings(len(tokenizer))

@torch.no_grad()
def translate(sentences, tag=None, batch_size=32):
    model.eval()
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = [f"{tag} {s}" if tag else s for s in sentences[i:i + batch_size]]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(model.device)
        out += tokenizer.batch_decode(model.generate(**enc, num_beams=4, max_length=64, forced_bos_token_id=JA), skip_special_tokens=True)
    return out

tests = ["I will go to Tokyo tomorrow.", "This book is very interesting.", "Do you know where the station is?", "He did not come to the party.",
         "We need to finish this work by Friday.", "It was raining when I left the house."]
before = translate(tests)
print(before)

['私は明日東京に行きます.', 'この本はとても興味深い.', '駅はどこか知ってる?', '彼はパーティーに来なかった.', '我々は金曜日までにこの仕事を終わらせなければならない.', '私は家を出ると雨が降っていました.']


In [4]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

enc = train.map(lambda b: tokenizer(b["en"], text_target=b["ja"], truncation=True, max_length=64), batched=True, remove_columns=["en", "ja", "style"])
args = Seq2SeqTrainingArguments(output_dir="mt-style", num_train_epochs=1, learning_rate=5e-5, warmup_steps=20,
                                per_device_train_batch_size=32, fp16=True, logging_steps=50, save_strategy="no", report_to="none", seed=0)
Seq2SeqTrainer(model=model, args=args, train_dataset=enc, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)).train()

Step,Training Loss
50,3.168778
100,2.621870
150,2.551264


TrainOutput(global_step=188, training_loss=2.724358335454413, metrics={'train_runtime': 28.8909, 'train_samples_per_second': 207.678, 'train_steps_per_second': 6.507, 'total_flos': 354863378595840.0, 'train_loss': 2.724358335454413, 'epoch': 1.0})

In [5]:
for en, b, p, q in zip(tests, before, translate(tests, "<polite>"), translate(tests, "<plain>")):
    print(f"{en}\n   学習前: {b}\n   <polite>: {p}\n   <plain> : {q}")

# 指定どおりの文体になった割合を、学習に使っていない文で測る
held_out = [ex["translation"]["en"] for ex in ds["test"] if 3 <= len(ex["translation"]["en"].split()) <= 20][:200]
for tag in ["<polite>", "<plain>"]:
    outputs = translate(held_out, tag)
    print(tag, "指定どおりの割合:", round(sum(style(o) == tag for o in outputs) / len(outputs), 3),
          "/ 判定できない文:", sum(style(o) is None for o in outputs))

I will go to Tokyo tomorrow.
   学習前: 私は明日東京に行きます.
   <polite>: 私は明日東京に行きます。
   <plain> : 私は明日東京に行きます。
This book is very interesting.
   学習前: この本はとても興味深い.
   <polite>: この本はとても興味深いものです。
   <plain> : この本はとても興味深い。
Do you know where the station is?
   学習前: 駅はどこか知ってる?
   <polite>: 駅の位置を知っていますか?
   <plain> : 駅はどこか知ってる?
He did not come to the party.
   学習前: 彼はパーティーに来なかった.
   <polite>: 彼はパーティーに来ませんでした。
   <plain> : 彼はパーティーには来なかった。
We need to finish this work by Friday.
   学習前: 我々は金曜日までにこの仕事を終わらせなければならない.
   <polite>: 私たちは金曜日までにこの作業を完了する必要があります。
   <plain> : 我々は金曜日までにこの仕事を完了しなければならない。
It was raining when I left the house.
   学習前: 私は家を出ると雨が降っていました.
   <polite>: 家を出ると雨でした。
   <plain> : 私は家を出たとき雨でした。
<polite> 指定どおりの割合: 0.8 / 判定できない文: 15
<plain> 指定どおりの割合: 0.775 / 判定できない文: 33


- 学習データの訳文を規則で分類し、その結果をタグとして原文の先頭に付けて学習する（サイドコンストレイント）。モデルは「タグと訳文の文体の対応」を学ぶので、翻訳時にタグを変えるだけで文体を切り替えられる。
- 対訳データに文体の注釈は要らない。訳文から自動的に判定できる性質なら、何でも制御の対象にできる（丁寧さ、態、長さ、分野など）。
- 6,000 文対・1 エポックなら T4 GPU で数分。新しく足したタグの埋め込みは乱数から始まるので、データが少ないと効きが不完全になる。指定どおりになった割合を必ず測る。
- 文体を変えると、訳の内容まで変わってしまうことがある。制御の成功率だけでなく、BLEU などで翻訳の質が落ちていないかも確かめる。
- LLM による翻訳なら、プロンプトで「です・ます調で」と指示するだけで同じことができる。専用の翻訳モデルを使う場面（速度、コスト、機密性）では、タグによる制御が今も有効。
- 1 つの言語の中で文体を書き換えるレシピは、要約と平易化の章の「文体を変換する（です・ます調、だ・である調、敬語）」にある。